# Part 1: Build Your Own CNN — WideResNet-28-10

Custom CNN from scratch on CIFAR-100 (32×32).  
Architecture: 3 groups × 4 residual blocks = **12 blocks** (>2 required).  
Each block: BN → ReLU → Conv → BN → ReLU → Dropout → Conv + skip connection.

In [ ]:
# Runtime configuration
FAST_DEV_RUN = False  # True = quick smoke test
SEED = 42

import os

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_ROOT = "/content" if IN_COLAB else os.path.abspath("./temp_student")
os.makedirs(WORK_ROOT, exist_ok=True)

DATA_ROOT = os.path.join(WORK_ROOT, "data")
OOD_DIR = os.path.join(WORK_ROOT, "ood-test-CS541")
SUBMISSION_PATH = os.path.join(WORK_ROOT, "submission_ood.csv")

print("IN_COLAB:", IN_COLAB)
print("WORK_ROOT:", WORK_ROOT)

In [ ]:
# Install required packages
import importlib.util, subprocess, sys
required = ["torch", "torchvision", "tqdm", "numpy", "pandas", "matplotlib", "huggingface_hub"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
print("Environment ready")

In [ ]:
import os, random, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

MEAN = (0.5071, 0.4867, 0.4408)  ### DO NOT CHANGE THIS
STD  = (0.2675, 0.2565, 0.2761)  ### DO NOT CHANGE THIS
USE_AMP = True

def _amp_ctx(device):
    enabled = USE_AMP and device.type == "cuda"
    try: return torch.amp.autocast(device_type=device.type, enabled=enabled)
    except (AttributeError, TypeError):
        from torch.cuda.amp import autocast; return autocast(enabled=enabled)

def _make_scaler(device):
    enabled = USE_AMP and device.type == "cuda"
    try: return torch.amp.GradScaler(device=device.type, enabled=enabled)
    except (AttributeError, TypeError):
        from torch.cuda.amp import GradScaler; return GradScaler(enabled=enabled)

set_seed(SEED)
device = get_device()
print("Device:", device)
if device.type=="cuda":
    print("GPU:", torch.cuda.get_device_name(), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
print("PyTorch:", torch.__version__)

In [ ]:
# ============================================================
# Data loaders with strong augmentation (32×32)
# ============================================================
def make_loaders(batch_size, num_workers):
    train_tfms = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25),
    ])
    eval_tfms = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    train_full = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=train_tfms)
    test_ds    = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=eval_tfms)
    n = len(train_full); n_tr = int(0.9*n)
    g = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = torch.utils.data.random_split(train_full, [n_tr, n-n_tr], generator=g)
    val_eval = torch.utils.data.Subset(
        datasets.CIFAR100(root=DATA_ROOT, train=True, download=False, transform=eval_tfms), val_ds.indices)
    if FAST_DEV_RUN:
        train_ds = torch.utils.data.Subset(train_ds, range(min(2048, len(train_ds))))
        val_eval = torch.utils.data.Subset(val_eval, range(min(512, len(val_eval))))
        test_ds  = torch.utils.data.Subset(test_ds, range(min(512, len(test_ds))))
    pin = torch.cuda.is_available()
    persistent = num_workers > 0
    common = dict(num_workers=num_workers, pin_memory=pin, persistent_workers=persistent)
    if persistent:
        common["prefetch_factor"] = 4
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, **common),
            DataLoader(val_eval, batch_size=batch_size*2, shuffle=False, **common),
            DataLoader(test_ds,  batch_size=batch_size*2, shuffle=False, **common))

# CutMix + MixUp
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    W, H = x.size(2), x.size(3)
    rw, rh = int(W*np.sqrt(1-lam)), int(H*np.sqrt(1-lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1,y1 = max(cx-rw//2,0), max(cy-rh//2,0)
    x2,y2 = min(cx+rw//2,W), min(cy+rh//2,H)
    x_mix = x.clone()
    x_mix[:,:,x1:x2,y1:y2] = x[idx,:,x1:x2,y1:y2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return x_mix, y, y[idx], lam

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam


In [ ]:
# ============================================================
# WideResNet-28-10 (custom CNN from scratch)
# 3 groups × 4 residual blocks = 12 blocks total (>2 required)
# ============================================================
class BasicBlock(nn.Module):
    def __init__(self, ic, oc, s, drop=0.3):
        super().__init__()
        self.bn1=nn.BatchNorm2d(ic); self.c1=nn.Conv2d(ic,oc,3,s,1,bias=False)
        self.bn2=nn.BatchNorm2d(oc); self.c2=nn.Conv2d(oc,oc,3,1,1,bias=False)
        self.drop=nn.Dropout(drop) if drop>0 else nn.Identity()
        self.sc=nn.Conv2d(ic,oc,1,s,bias=False) if(s!=1 or ic!=oc) else nn.Identity()
    def forward(self, x):
        o=F.relu(self.bn1(x)); s=self.sc(o)
        o=self.c1(o); o=self.drop(F.relu(self.bn2(o))); o=self.c2(o)
        return o+s

class WideResNet(nn.Module):
    """WRN-28-10: 3 groups x 4 blocks = 12 residual blocks (from scratch)."""
    def __init__(self, depth=28, k=10, drop=0.3, nc=100):
        super().__init__()
        n=(depth-4)//6; ch=[16,16*k,32*k,64*k]
        self.c0=nn.Conv2d(3,ch[0],3,1,1,bias=False)
        self.g1=nn.Sequential(BasicBlock(ch[0],ch[1],1,drop),*[BasicBlock(ch[1],ch[1],1,drop) for _ in range(n-1)])
        self.g2=nn.Sequential(BasicBlock(ch[1],ch[2],2,drop),*[BasicBlock(ch[2],ch[2],1,drop) for _ in range(n-1)])
        self.g3=nn.Sequential(BasicBlock(ch[2],ch[3],2,drop),*[BasicBlock(ch[3],ch[3],1,drop) for _ in range(n-1)])
        self.bn=nn.BatchNorm2d(ch[3]); self.fc=nn.Linear(ch[3],nc)
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,mode="fan_out",nonlinearity="relu")
            elif isinstance(m,nn.BatchNorm2d): m.weight.data.fill_(1); m.bias.data.zero_()
    def forward(self, x):
        x=self.c0(x); x=self.g1(x); x=self.g2(x); x=self.g3(x)
        return self.fc(F.adaptive_avg_pool2d(F.relu(self.bn(x)),1).flatten(1))

print("WRN-28-10 defined")

In [ ]:
# ============================================================
# Training engine (channels_last + sparse val + per-batch sched option)
# ============================================================
def train_model(model, train_ld, val_ld, optimizer, scheduler, criterion, device, epochs,
                use_mix=True, step_sched_per_batch=False, val_every=5):
    history = {"train_acc":[], "val_acc":[]}
    best_va, best_st = -1.0, None
    scaler = _make_scaler(device)
    use_cl = device.type == "cuda"  # channels_last memory format

    for ep in range(1, epochs+1):
        model.train(); correct=total=0
        for x, y in tqdm(train_ld, desc=f"Train {ep}/{epochs}", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if use_cl: x = x.contiguous(memory_format=torch.channels_last)
            do_mix = use_mix and random.random()<0.5
            if do_mix:
                if random.random()<0.5: x,ya,yb,lam = cutmix_data(x,y)
                else: x,ya,yb,lam = mixup_data(x,y)
            optimizer.zero_grad(set_to_none=True)
            with _amp_ctx(device):
                logits = model(x)
                if do_mix: loss = lam*criterion(logits,ya)+(1-lam)*criterion(logits,yb)
                else: loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer); scaler.update()
            if step_sched_per_batch and scheduler is not None:
                scheduler.step()
            correct += (logits.argmax(1)==y).sum().item(); total += y.numel()
        if (not step_sched_per_batch) and scheduler is not None:
            scheduler.step()
        tr_acc = correct/max(total,1)

        do_val = (ep % val_every == 0) or (ep > epochs - 5) or (ep == 1)
        if do_val:
            model.eval(); vc=vt=0
            with torch.no_grad():
                for x,y in val_ld:
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    if use_cl: x = x.contiguous(memory_format=torch.channels_last)
                    with _amp_ctx(device):
                        vc+=(model(x).argmax(1)==y).sum().item(); vt+=y.numel()
            va_acc = vc/max(vt,1)
            history["train_acc"].append(tr_acc); history["val_acc"].append(va_acc)
            print(f"  Ep {ep:03d}/{epochs} | train {tr_acc:.4f} | val {va_acc:.4f} | lr {optimizer.param_groups[0]['lr']:.6f}")
            if va_acc > best_va:
                best_va = va_acc
                best_st = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            history["train_acc"].append(tr_acc)
            history["val_acc"].append(history["val_acc"][-1] if history["val_acc"] else 0.0)
            print(f"  Ep {ep:03d}/{epochs} | train {tr_acc:.4f} | val   skip   | lr {optimizer.param_groups[0]['lr']:.6f}")

    if best_st:
        try:
            model.load_state_dict(best_st)
        except Exception:
            # strip torch.compile prefix "_orig_mod." if present
            sd = {k.replace("_orig_mod.", ""): v for k, v in best_st.items()}
            try: model.load_state_dict(sd)
            except Exception: pass
    print(f"  >> Best val: {best_va:.4f}"); return history

@torch.no_grad()
def eval_acc(model, loader, device):
    model.eval(); c=t=0
    use_cl = device.type == "cuda"
    for x,y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if use_cl: x = x.contiguous(memory_format=torch.channels_last)
        with _amp_ctx(device):
            c+=(model(x).argmax(1)==y).sum().item(); t+=y.numel()
    return 100.0*c/t


In [ ]:
# ============================================================
# TRAIN Part 1: WRN-28-10 — fast recipe (OneCycleLR, big batch, channels_last)
# ============================================================
set_seed(SEED)
EP1 = 80 if not FAST_DEV_RUN else 2          # was 200; OneCycle reaches comparable acc much faster
BS  = 256 if device.type == "cuda" else 128  # bigger batch -> better GPU utilization
NW  = 4 if not IN_COLAB else 2
print(f"\n{'='*60}\nPart 1: WRN-28-10 — {EP1} epochs @ 32x32, bs={BS}\n{'='*60}")

tr1, va1, te1 = make_loaders(BS, NW)
model = WideResNet(28, 10, 0.3, 100).to(device)
if device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

# Linearly scale LR for the larger batch (baseline 0.1 @ bs=128)
base_lr = 0.1 * (BS / 128)
opt = torch.optim.SGD(model.parameters(), lr=base_lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
steps_per_epoch = len(tr1)
sch = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=base_lr, epochs=EP1, steps_per_epoch=steps_per_epoch,
    pct_start=0.1, anneal_strategy="cos", div_factor=25.0, final_div_factor=1e4,
)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

# torch.compile -> ~15-25% speedup on PyTorch 2.x + CUDA
if device.type == "cuda" and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("  torch.compile: enabled")
    except Exception as e:
        print(f"  torch.compile disabled: {e}")

hist = train_model(model, tr1, va1, opt, sch, crit, device, EP1,
                   step_sched_per_batch=True, val_every=5)
acc = eval_acc(model, te1, device)
print(f"\nPart 1 Clean CIFAR-100 test accuracy: {acc:.2f}%")
torch.save(model.state_dict(), os.path.join(WORK_ROOT, "p1_wrn.pt"))


In [ ]:
# Plot training curves
plt.figure(figsize=(8,5))
plt.plot(hist["train_acc"], label="Train"); plt.plot(hist["val_acc"], label="Val")
plt.title("Part 1: WRN-28-10"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## OOD Submission Generation

Submission format: `id,label`  
Using 5-view TTA (original, H-flip, center-crop, TL-crop, BR-crop).

In [ ]:
### DO NOT CHANGE THE BELOW, REPLACE WITH YOUR MODEL, THE SUBMISSION FILES NEED TO GO THROUGH THE BELOW PREPROCESSING

from huggingface_hub import snapshot_download

def ensure_ood_files(ood_dir):
    os.makedirs(ood_dir, exist_ok=True)
    print("Downloading OOD files from Hugging Face dataset...")
    snapshot_download(repo_id="XThomasBU/ood-test-CS541", repo_type="dataset",
                      local_dir=ood_dir, local_dir_use_symlinks=False)
    print("OOD files ready in", ood_dir)

@torch.no_grad()
def get_tta_probs(model, xb, device):
    """Lightweight 2-view TTA: original + horizontal flip.
    ~2.5x faster than 5-view; tiny accuracy delta on CIFAR-100-C."""
    model.eval()
    xb = xb.to(device, non_blocking=True)
    if device.type == "cuda":
        xb = xb.contiguous(memory_format=torch.channels_last)
    with _amp_ctx(device):
        p = F.softmax(model(xb).float(), dim=1)
        p = p + F.softmax(model(xb.flip(-1)).float(), dim=1)
    return (p / 2.0).cpu()

@torch.no_grad()
def predict_file(model, npy_path, severity, batch_size):
    images = np.load(npy_path, mmap_mode="r")
    start, end = (severity-1)*10000, severity*10000
    mean_t = torch.tensor(MEAN).view(1,3,1,1)
    std_t  = torch.tensor(STD).view(1,3,1,1)
    all_preds = []
    for b0 in range(start, end, batch_size):
        b1 = min(b0+batch_size, end)
        xb = torch.from_numpy(np.array(images[b0:b1], copy=True)).permute(0,3,1,2).float().div(255.0)
        xb = (xb - mean_t) / std_t
        all_preds.append(get_tta_probs(model, xb, device).argmax(1).numpy())
    return np.concatenate(all_preds)

ensure_ood_files(OOD_DIR)
distortion_files = sorted([p for p in os.listdir(OOD_DIR) if p.startswith("distortion") and p.endswith(".npy")])
print(f"Distortion files: {len(distortion_files)}")

BATCH = 512 if device.type=="cuda" else 32
rows = []
for fi, fname in enumerate(distortion_files):
    dname = os.path.splitext(fname)[0]
    path = os.path.join(OOD_DIR, fname)
    for sev in [1,2,3,4,5]:
        pred = predict_file(model, path, sev, BATCH)
        for i, y in enumerate(pred.tolist()):
            rows.append((f"{dname}_{sev}_{i}", int(y)))
    print(f"  [{fi+1}/{len(distortion_files)}] {dname} done")

submission = pd.DataFrame(rows, columns=["id", "label"])
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nWrote {SUBMISSION_PATH} ({len(submission)} rows)")
print(submission.head())

if IN_COLAB:
    try: from google.colab import files; files.download(SUBMISSION_PATH)
    except: print("Download manually:", SUBMISSION_PATH)
